In [1]:
import numpy as np
import networkx as nx
import pandas as pd
from scipy import stats
from itertools import combinations
import multiprocessing as mp
import os
import glob
import warnings
warnings.filterwarnings("ignore")

# Utility
def min_max_norm(x: np.ndarray) -> np.ndarray:
    lo, hi = np.nanmin(x), np.nanmax(x)
    if hi == lo:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)


# Clustered seeding
def clustered_seeding(seeds: list, g: nx.Graph, seeds_needed: int, rng: np.random.Generator, deterministic: bool = False, exclude: set = None) -> list:
    if exclude is None:
        exclude = set()
    possible = set()
    for s in seeds:
        possible.update(g.neighbors(s))
    to_add = [v for v in possible if v not in seeds and v not in exclude]

    if seeds_needed > len(to_add):
        return list(set(seeds) | possible)
    else:
        if deterministic:
            extra = sorted(to_add)[:seeds_needed]
        else:
            extra = rng.choice(to_add, size=seeds_needed, replace=False).tolist()
        return list(set(seeds) | set(extra))


# Simple centrality measures
def get_simple_centralities(g: nx.Graph) -> pd.DataFrame:
    """
    degree، betweenness، eigenvector و collective influence (d=3)
    """
    nodes = list(g.nodes())
    degree = dict(g.degree())

    betweenness = nx.betweenness_centrality(g, normalized=False)

    try:
        eigen = nx.eigenvector_centrality(g, max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        eigen = {v: 0.0 for v in nodes}

    # Collective Influence at d=3: CI(i) = (deg(i)-1) * Σ(deg(j)-1)  for j at dist=3
    percolation = {}
    for v in nodes:
        ball_3 = [u for u, d in nx.single_source_shortest_path_length(g, v, cutoff=3).items() if d == 3]
        ci = (degree[v] - 1) * sum(max(degree[u] - 1, 0) for u in ball_3)
        percolation[v] = ci

    return pd.DataFrame({
        "seed":        nodes,
        "degree":      [degree[v]      for v in nodes],
        "betweenness": [betweenness[v] for v in nodes],
        "eigen":       [eigen[v]       for v in nodes],
        "percolation": [percolation[v] for v in nodes],
    })


# Core: diffusion + complex path length
def get_complex(seed: int, N: int, g: nx.Graph, gmat: np.ndarray, thresholds: np.ndarray, num_seeds_to_add: np.ndarray, random_seed: int = None, deterministic: bool = False, verbose: bool = False) -> dict:
    rng = np.random.default_rng(random_seed)

    # ── ساخت seed set ────────────────────────────────────────────────────────
    num_seeds_i = int(num_seeds_to_add[seed])
    neighbours  = list(g.neighbors(seed))

    if num_seeds_i == 0:
        initial_seeds = [seed]
    else:
        needed = num_seeds_i - len(neighbours)
        if needed > 0:
            # clustered seeding: از همسایه‌های همسایه‌ها استفاده می‌کند
            neighbours = clustered_seeding(neighbours, g, needed, rng, deterministic=deterministic, exclude={seed})
            needed = num_seeds_i - len(neighbours)
            # اگر باز هم کم باشد، همه را بگیر
            if needed > 0:
                initial_seeds = [seed] + neighbours
            else:
                if deterministic:
                    chosen = sorted(neighbours)[:num_seeds_i]
                else:
                    chosen = rng.choice(neighbours, size=min(num_seeds_i, len(neighbours)), replace=False).tolist()
                initial_seeds = [seed] + chosen
        elif len(neighbours) > 1:
            if deterministic:
                chosen = sorted(neighbours)[:num_seeds_i]
            else:
                chosen = rng.choice(neighbours, size=num_seeds_i, replace=False).tolist()
            initial_seeds = [seed] + chosen
        else:
            initial_seeds = [seed] + neighbours

    # ── شبیه‌سازی diffusion ──────────────────────────────────────────────────
    activated = np.zeros(N, dtype=bool)
    for s in initial_seeds:
        if 0 <= s < N:
            activated[s] = True

    spread = True
    while spread:
        prev_count = int(activated.sum())
        active_influence = gmat[:, :] * activated[:, np.newaxis]   # N×N
        neighbour_count  = active_influence.sum(axis=0)            # length N

        newly_activated = (neighbour_count >= thresholds) & (~activated)
        activated |= newly_activated

        if int(activated.sum()) == prev_count:
            spread = False

    num_adopters = int(activated.sum())

    # ── complex path length : PLci طبق معادله‌ی (5) مقاله ────────────────────
    activated_nodes = np.where(activated)[0].tolist()
    sub = g.subgraph(activated_nodes)

    closed_nb = set([seed] + list(g.neighbors(seed)))      # N[i] کامل
    closed_neighborhood_size = len(closed_nb)               # |N[i]|
    denom = N - closed_neighborhood_size

    if seed in sub and denom > 0:
        dist_map = nx.single_source_shortest_path_length(sub, seed)
        total_distance = sum(d for node, d in dist_map.items() if node not in closed_nb)
        PLci = float(total_distance) / denom
    else:
        total_distance = 0
        PLci = 0.0

    if verbose:
        print(f"  seed={seed}  initial_seeds={sorted(initial_seeds)}  "
              f"activated={sorted(activated_nodes)}  N[seed]={sorted(closed_nb)}  "
              f"denom={denom}  total_distance={total_distance}  PLci={PLci:.4f}")

    return {
        "seed":            seed,
        "N":               N,
        "num_neigh_seeds": num_seeds_i,
        "num_adopters":    num_adopters,
        "PLci":            PLci,
    }


def neighbours_original(g: nx.Graph, seed: int) -> list:
    return [seed] + list(g.neighbors(seed))


# Worker برای multiprocessing
def _worker(args):
    seed, N, g, gmat, thresholds, num_seeds_to_add, rseed, deterministic = args
    return get_complex(seed, N, g, gmat, thresholds, num_seeds_to_add, random_seed=rseed, deterministic=deterministic)


# اجرای مدل (sequential یا parallel)
def run_model(g: nx.Graph, gmat: np.ndarray, thresholds: np.ndarray, num_seeds_to_add: np.ndarray, n_cores: int = 1, deterministic: bool = False, verbose: bool = False) -> pd.DataFrame:
    N     = gmat.shape[0]
    nodes = list(g.nodes())
    rseeds = list(range(len(nodes)))   # برای reproducibility

    if n_cores == 1:
        results = [
            get_complex(s, N, g, gmat, thresholds, num_seeds_to_add, rs, deterministic=deterministic, verbose=verbose)
            for s, rs in zip(nodes, rseeds)
        ]
    else:
        arg_list = [
            (s, N, g, gmat, thresholds, num_seeds_to_add, rs, deterministic)
            for s, rs in zip(nodes, rseeds)
        ]
        with mp.Pool(processes=n_cores) as pool:
            results = pool.map(_worker, arg_list)

    df = pd.DataFrame(results)
    df["PLci_norm"] = min_max_norm(df["PLci"].values)
    return df


# Pipeline کامل برای یک گراف
def analyse_graph(g: nx.Graph, T_type: str = "abs", threshold_val: float = 3.0, n_cores: int = 1, deterministic: bool = False, min_size: int = 10, verbose: bool = False) -> pd.DataFrame:
    """
    pipeline کامل:
      1. ماتریس مجاورت
      2. تعیین threshold
      3. محاسبه centrality ساده
      4. اجرای مدل complex path length
      5. ادغام نتایج
    """
    if len(g) <= min_size:
        return pd.DataFrame()

    N    = len(g)
    gmat = np.array(nx.to_numpy_array(g), dtype=np.float32)

    thresholds = np.full(N, threshold_val)
    if T_type == "frac":
        degrees    = np.array([d for _, d in g.degree()])
        thresholds = np.round(thresholds * degrees).astype(float)

    num_seeds_to_add = np.maximum(thresholds - 1, 0)
    thresholds[thresholds <= 0] = 1

    print(f"  threshold={threshold_val}  T_type={T_type}  "
          f"avg_degree={np.mean([d for _,d in g.degree()]):.3f}")

    simple_df = get_simple_centralities(g)
    model_df  = run_model(g, gmat, thresholds, num_seeds_to_add, n_cores=n_cores, deterministic=deterministic, verbose=verbose)

    merged = model_df.merge(simple_df, on="seed")
    merged["threshold"] = threshold_val
    merged["T_type"]    = T_type

    print(f"  avg_adopters={merged['num_adopters'].mean():.3f}  "
          f"max_adopters={merged['num_adopters'].max()}  "
          f"avg_PLci={merged['PLci'].mean():.3f}")
    return merged


# مقایسه استراتژی‌ها
def compare_strategies(df: pd.DataFrame, graph_id: int) -> pd.DataFrame:
    """
    بهترین seed طبق هر معیار centrality را انتخاب می‌کند.
    """
    strategies = {
        "Complex":     "PLci_norm",
        "betweenness": "betweenness",
        "degree":      "degree",
        "eigen":       "eigen",
        "percolation": "percolation",
    }
    rows = []
    for label, col in strategies.items():
        best      = df.nlargest(1, col).sample(1)
        row       = best.copy()
        row["top"]   = label
        row["graph"] = graph_id
        rows.append(row)
    return pd.concat(rows, ignore_index=True)


# آزمون Wilcoxon زوجی
#def pairwise_wilcoxon(comparison_df: pd.DataFrame, outcome_col: str = "num_adopters", group_col: str = "top") -> pd.DataFrame:
    """
    آزمون Wilcoxon signed-rank زوجی بین همه جفت استراتژی‌ها
    (معادل R: pairwise.wilcox.test(..., paired=TRUE, p.adjust="none"))
    """
    groups  = comparison_df[group_col].unique()
    records = []
    for g1, g2 in combinations(groups, 2):
        df1 = (comparison_df[comparison_df[group_col] == g1]
               [["graph", outcome_col]].set_index("graph"))
        df2 = (comparison_df[comparison_df[group_col] == g2]
               [["graph", outcome_col]].set_index("graph"))
        paired = df1.join(df2, lsuffix="_1", rsuffix="_2").dropna()
        if len(paired) < 2:
            continue
        x, y = paired[f"{outcome_col}_1"].values, paired[f"{outcome_col}_2"].values
        if np.all(x == y):          # Wilcoxon نیاز به تفاوت دارد
            continue
        stat, p = stats.wilcoxon(x, y, alternative="two-sided")
        records.append({"group1": g1, "group2": g2, "statistic": stat, "p_value": round(p, 4)})
    return pd.DataFrame(records)


# بارگذاری شبکه‌های AddHealth
def load_addhealth_networks(directory: str) -> list:
    """
    همه فایل‌های CSV در پوشه را به عنوان ماتریس مجاورت بارگذاری می‌کند.
    فرمت مورد انتظار: ماتریس مربعی بدون header و بدون ستون index
    """
    files = sorted(glob.glob(os.path.join(directory, "*.csv")))
    if not files:
        raise FileNotFoundError(f"هیچ فایل CSV در این مسیر یافت نشد: {directory}")

    networks = []
    for fpath in files:
        try:
            mat = pd.read_csv(fpath, header=0).values.astype(np.float32)
            g   = nx.from_numpy_array(mat)
            networks.append((mat, g))
            print(f"  ✓ {os.path.basename(fpath)}  |  N={mat.shape[0]}  "
                  f"|  edges={g.number_of_edges()}")
        except Exception as e:
            print(f"  ✗ {os.path.basename(fpath)}: {e}")
    return networks


# تولید شبکه‌های Holme-Kim
def generate_holme_kim_networks(num_graphs: int = 20, N: int = 200, m: int = 4, p: float = 0.5) -> list:
    """
    شبکه‌های scale-free با clustering قابل تنظیم (Holme & Kim 2002)
    """
    networks = []
    for i in range(num_graphs):
        g   = nx.powerlaw_cluster_graph(N, m, p, seed=i)
        mat = np.array(nx.to_numpy_array(g), dtype=np.float32)
        networks.append((mat, g))
    return networks


# Pipeline اصلی
def main(
    source:         str   = "holme_kim",  # "holme_kim" | "addhealth" | "manual_example"
    addhealth_dir:  str   = "",           # مسیر پوشه CSVبرای AddHealth
    num_graphs:     int   = 20,
    N:              int   = 200,
    T_type:         str   = "abs",        # "abs" | "frac"
    threshold_val:  float = 3.0,
    n_cores:        int   = 1,
    deterministic:  bool  = False,
):
    
    # ── بارگذاری / تولید شبکه‌ها ────────────────────────────────────────────
    if source == "addhealth":
        print(f"\nبارگذاری شبکه‌های AddHealth از: {addhealth_dir}")
        networks = load_addhealth_networks(addhealth_dir)
    else:
        print(f"\nتولید {num_graphs} گراف Holme-Kim (N={N}) …")
        networks = generate_holme_kim_networks(num_graphs, N)

    comparison_df = pd.DataFrame()

    for i, (gmat, g) in enumerate(networks):
        print(f"\nگراف {i+1}/{len(networks)}  |  N={gmat.shape[0]}  |  cores={n_cores}")
        df = analyse_graph(g, T_type=T_type, threshold_val=threshold_val, n_cores=n_cores, deterministic=deterministic)
        if df.empty:
            continue
        top_df        = compare_strategies(df, graph_id=i + 1)
        comparison_df = pd.concat([comparison_df, top_df], ignore_index=True)

    if comparison_df.empty:
        print("نتیجه‌ای برای نمایش وجود ندارد.")
        return comparison_df

    # ── خلاصه ────────────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("نتایج هر گراف جداگانه")
    print("="*70)

    for graph_id in sorted(comparison_df["graph"].unique()):
        print(f"\n📊 گراف شماره {graph_id}")
        subset = comparison_df[comparison_df["graph"] == graph_id]
        means_per_graph = (subset
                          .groupby("top")["num_adopters"]
                          .mean()
                          .sort_values(ascending=False))
        print(means_per_graph.to_string())

    print("\n" + "="*70)
    print("نتایج کلی (میانگین تمام گراف‌ها)")
    print("="*70)
    means = (comparison_df
             .groupby("top")["num_adopters"]
             .mean()
             .sort_values(ascending=False))
    print(means.to_string())

    #print("\n" + "="*70)
    #print("آزمون Wilcoxon زوجی")
    #print("="*70)
    #wilcox = pairwise_wilcoxon(comparison_df)
    #if not wilcox.empty:
    #    print(wilcox.to_string(index=False))
    #else:
    #    print("  (تعداد مشاهدات زوجی کافی نیست — تعداد گراف‌ها را افزایش دهید)")

    return comparison_df


# Entry point
if __name__ == "__main__":
    for threshold in [3.0, 4.0]:
        results_ah = main(
            source        = "addhealth",
            addhealth_dir = "/mnt/f/test1/",   # ← مسیر ویندوزی F:\test در WSL
            T_type        = "abs",
            threshold_val = threshold,
            n_cores       = 1,
        )


بارگذاری شبکه‌های AddHealth از: /mnt/f/test1/
  ✓ addhealth_net_1.csv  |  N=69  |  edges=220
  ✓ addhealth_net_10.csv  |  N=678  |  edges=2795
  ✓ addhealth_net_11.csv  |  N=411  |  edges=1590
  ✓ addhealth_net_12.csv  |  N=581  |  edges=2805
  ✓ addhealth_net_13.csv  |  N=652  |  edges=2350
  ✓ addhealth_net_14.csv  |  N=562  |  edges=2687
  ✓ addhealth_net_15.csv  |  N=562  |  edges=2688
  ✓ addhealth_net_16.csv  |  N=1062  |  edges=4391
  ✓ addhealth_net_17.csv  |  N=778  |  edges=3462
  ✓ addhealth_net_18.csv  |  N=1095  |  edges=4144
  ✓ addhealth_net_2.csv  |  N=103  |  edges=348
  ✓ addhealth_net_20.csv  |  N=492  |  edges=2096
  ✓ addhealth_net_21.csv  |  N=910  |  edges=4075
  ✓ addhealth_net_22.csv  |  N=377  |  edges=1531
  ✓ addhealth_net_23.csv  |  N=612  |  edges=2449
  ✓ addhealth_net_24.csv  |  N=667  |  edges=2897
  ✓ addhealth_net_25.csv  |  N=849  |  edges=3003
  ✓ addhealth_net_27.csv  |  N=551  |  edges=2066
  ✓ addhealth_net_28.csv  |  N=1152  |  edges=2530
  ✓ a